# 08 – Demo para la revisión en vivo
Notebook corto: no entrena nada. Usa los modelos y predicciones ya guardados en el repositorio.

**Antes de entrar:** Entorno de ejecución → GPU T4 → *Ejecutar todas*. La descarga del dataset tarda ~2 minutos la primera vez; el resto es inmediato.

| Sección | Qué muestra |
|---|---|
| (a) | Cargar los datos y ver ejemplos |
| (b) | Cargar un `.keras` y clasificar una imagen con sus probabilidades |
| (c) | Recalcular macro-F1 y matriz desde los `.npy` guardados |
| (d) | Cambiar una transformación (brillo o rotación) y ver el efecto |
| (e) | Mostrar y explicar un error concreto |

In [ ]:
# 1) Librerías y reglas del protocolo (iguales en todos los notebooks)
!pip install -q importlib_resources   # sin esto tfds.load falla en Colab

import os, time, json, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

SEED = 42      # semilla fija: así los resultados se pueden repetir
IMG = 224      # tamaño de entrada que esperan MobileNetV2 y ResNet50V2
BATCH = 32     # fotos por lote
EPOCHS = 10    # presupuesto máximo de épocas, igual para todos los experimentos
AUTOTUNE = tf.data.AUTOTUNE
tf.keras.utils.set_random_seed(SEED)

RESULTS = "results"
os.makedirs(RESULTS, exist_ok=True)

# Lo que ya está subido al repositorio (pesos de E1, índices excluidos, etc.) se descarga de GitHub
REPO_RAW = "https://raw.githubusercontent.com/cristianh1818/Proyecto-Yuca-Machine-learning/main/results/"

def traer(archivo):
    """Devuelve la ruta de results/<archivo>. Si no está en Colab, lo baja del repositorio."""
    ruta = f"{RESULTS}/{archivo}"
    if not os.path.exists(ruta):
        r = requests.get(REPO_RAW + archivo, timeout=600)
        r.raise_for_status()
        with open(ruta, "wb") as f:
            f.write(r.content)
    return ruta

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## (a) Cargar los datos y mostrar ejemplos
Solo validación, con las mismas fotos excluidas que en todos los experimentos.

In [ ]:
ds_val, info = tfds.load("cassava", split="validation", as_supervised=True, with_info=True)
CLASES = info.features["label"].names
N_CLASES = len(CLASES)

excl = json.load(open(traer("indices_excluidos_final.json")))["validation"]
quitar = tf.constant(sorted(excl), dtype=tf.int64)
ds_val = ds_val.enumerate().filter(
    lambda i, xy: tf.logical_not(tf.reduce_any(tf.equal(i, quitar)))).map(lambda i, xy: xy)

# Fotos de validación ya redimensionadas a 224x224 (en memoria)
x_val, y_val = [], []
for img, y in tfds.as_numpy(ds_val.map(lambda x, y: (tf.cast(tf.image.resize(x, (IMG, IMG)), tf.uint8), y))):
    x_val.append(img); y_val.append(int(y))
x_val, y_val = np.array(x_val), np.array(y_val)
print("Fotos de validación:", len(y_val), "| por clase:", {c: int((y_val == i).sum()) for i, c in enumerate(CLASES)})

fig, axs = plt.subplots(1, N_CLASES, figsize=(16, 3.5))
for i, c in enumerate(CLASES):
    k = np.where(y_val == i)[0][0]
    axs[i].imshow(x_val[k]); axs[i].set_title(c); axs[i].axis("off")
plt.tight_layout(); plt.show()

## (b) Cargar un modelo guardado y clasificar una imagen
Cambiar `MODELO` o `I`.

In [ ]:
MODELO = "E1_mobilenetv2_congelado"   # también: "E0_baseline", "E2_mobilenetv2_finetuning"
I = 40                                # índice de la foto de validación

modelo = tf.keras.models.load_model(traer(f"{MODELO}.keras"))

def clasificar(img_uint8, titulo=""):
    p = modelo.predict(img_uint8[None].astype("float32"), verbose=0)[0]
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
    axs[0].imshow(img_uint8.astype("uint8")); axs[0].axis("off"); axs[0].set_title(titulo)
    axs[1].barh(CLASES, p, color=["tab:green" if j == p.argmax() else "tab:gray" for j in range(N_CLASES)])
    axs[1].set_xlim(0, 1); axs[1].set_title("probabilidades")
    plt.tight_layout(); plt.show()
    return p

p = clasificar(x_val[I], f"real: {CLASES[y_val[I]]}")
print("Predicción:", CLASES[p.argmax()], f"({p.max():.2f})")

## (c) Recalcular macro-F1 y matriz desde las predicciones guardadas
No se reentrena: se usan los `.npy` del repositorio.

In [ ]:
yp = np.load(traer(f"{MODELO}_ypred_validacion.npy"))
print(f"{MODELO} – validación")
print("Accuracy:", round(accuracy_score(y_val, yp), 4))
print("Macro-F1:", round(f1_score(y_val, yp, average="macro"), 4))
print("F1 por clase:", dict(zip(CLASES, f1_score(y_val, yp, average=None).round(3))))
print(classification_report(y_val, yp, target_names=CLASES, digits=3, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_val, yp, display_labels=CLASES, cmap="Greens", colorbar=False)
plt.title(f"{MODELO} – validación"); plt.show()

## (d) Cambiar una transformación y ver el efecto
Cambiar `BRILLO` (1.0 = sin cambio) o `ROTACION` (grados). Se aplica a una foto y a 300 fotos de validación.

In [ ]:
from scipy.ndimage import rotate
BRILLO = 1.3      # <1 oscurece, >1 aclara
ROTACION = 0      # grados, por ejemplo 30
N = 300           # fotos de validación para medir el efecto (rápido)

def transformar(lote):
    out = np.clip(lote.astype("float32") * BRILLO, 0, 255)
    if ROTACION:
        out = np.stack([rotate(im, ROTACION, reshape=False, mode="reflect") for im in out])
    return out

clasificar(transformar(x_val[I:I + 1])[0], f"transformada (brillo×{BRILLO}, rot {ROTACION}°)")

idx = np.random.default_rng(SEED).choice(len(y_val), N, replace=False)
p_orig = modelo.predict(x_val[idx].astype("float32"), verbose=0).argmax(1)
p_trans = modelo.predict(transformar(x_val[idx]), verbose=0).argmax(1)
print(f"Macro-F1 en {N} fotos – original: {f1_score(y_val[idx], p_orig, average='macro'):.4f} "
      f"| transformadas: {f1_score(y_val[idx], p_trans, average='macro'):.4f}")
print("Predicciones que cambiaron:", int((p_orig != p_trans).sum()), "de", N)

## (e) Mostrar y explicar un error concreto
Ejemplo del registro de errores de E1: **#129**, real *cbsd*, predicho *cbb* con confianza 0,94. La hoja está amarilla con manchas cafés necróticas y bordes secos, rasgos más típicos de CBB: es un candidato fuerte a **etiqueta errónea** o coinfección, coherente con las 18 fotos con etiquetas contradictorias del EDA. Otros: #772 (cmd → cgm, 0,88, moteado parecido), #787 (cbb → cbsd, mancha pequeña a contraluz).

In [ ]:
ERROR = 129    # cambiar por otro índice de registro_errores.md (p. ej. 772, 787, 1251)
p = clasificar(x_val[ERROR], f"#{ERROR} real: {CLASES[y_val[ERROR]]}")
print("Predicción:", CLASES[p.argmax()], f"(confianza {p.max():.2f})")
print("Segunda opción:", CLASES[np.argsort(p)[-2]], f"({np.sort(p)[-2]:.2f})")